In [ ]:
from pathlib import Path
from shutil import rmtree
from json import dumps
from typing import Literal
from pickle import Pickler
from gzip import open as gz_open
from joblib import Memory

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, minmax_scale
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.metrics import make_scorer, f1_score


RANDOM_STATE = 42
RANDOM_RNG = np.random.default_rng(42)
VIF_THRESHOLD = 10.0

# Data

In [ ]:
def get_vif_removal_features(threshold: float) -> frozenset[str]:
    vif_values = Path("../datasense/dataset/vif_values.csv")
    vif_values = pd.read_csv(vif_values)
    vif_values = vif_values[vif_values["vif"] >= threshold]
    return frozenset(vif_values["feature"])


def get_transformer(data: pd.DataFrame, vif_threshold: float = VIF_THRESHOLD):
    number_selector = make_column_selector(dtype_include="number")
    bool_selector = make_column_selector(dtype_include="bool")
    first_transformer = make_column_transformer(
        (OneHotEncoder(sparse_output=False), ["device_type"]),
        ("passthrough", number_selector),
        ("passthrough", bool_selector),
        remainder="drop",
        verbose_feature_names_out=False,
    )
    _ = first_transformer.fit_transform(data)
    feature_names = first_transformer.get_feature_names_out()
    vif_removal = get_vif_removal_features(vif_threshold)
    vif_indexes = [i for i, ft in enumerate(feature_names) if ft in vif_removal]
    second_transformer = make_column_transformer(
        ("drop", vif_indexes),
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    return make_pipeline(first_transformer, second_transformer)


def prepare_experiment(
    target: Literal["binary", "condensed", "granular"],
    overwrite: bool = False,
):
    experiment = {}
    dataset_path = Path("../datasense/dataset/")
    for split in ["train", "test"]:
        split_data = dataset_path / f"datasense_1sec_{split}.parquet"
        split_data = pd.read_parquet(split_data)
        if target == "binary":
            y_split = split_data["label1"]
        elif target == "condensed":
            y_split = split_data["label2"]
        elif target == "granular":
            y_split = split_data["label4"]
        else:
            raise ValueError("Unknown target")
        experiment[split] = (split_data, y_split)
    X_train, y_train = experiment["train"]
    experiment["classes"] = sorted(y_train.unique())
    experiment["transformer"] = get_transformer(X_train)
    results_dir = Path(f"results/decision_tree__{target}")
    if results_dir.is_dir():
        if overwrite:
            rmtree(results_dir, ignore_errors=True)
        else:
            raise RuntimeError(
                "Experiment already executed and overwrite set to False"
            )
    results_dir.mkdir(exist_ok=True, parents=True)
    cache_dir = results_dir / ".cache"
    cache_dir.mkdir()
    experiment["results_dir"] = results_dir
    experiment["cache_dir"] = Memory(location=cache_dir, verbose=0)
    return experiment

In [ ]:
experiment = prepare_experiment(target="granular", overwrite=True)

X_train, y_train = experiment["train"]
class_names = experiment["classes"]
num_classes = len(class_names)
X_test, y_test = experiment["test"]
results_dir = experiment["results_dir"]
cache_dir = experiment["cache_dir"]

# Experiment

## Training pipe

In [ ]:
model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=RANDOM_RNG,
)

In [ ]:
train_pipe = Pipeline(
    steps=[
        ("transformer", experiment["transformer"]),
        ("model", model),
    ],
    memory=cache_dir,
    verbose=False,
)

## Grid seach cross-validation

In [ ]:
param_grid = {
    "model__criterion": ["gini", "entropy"],
    # `min_samples_leaf` : offer a bit more of regularization for the training
    "model__min_samples_leaf": [3, 5, 7],
    # `max_depth`      : is too restrictive a measure of complexity
    # `max_leaf_nodes` : represents the level of partitioning of the training data
    # `max_leaf_nodes` : too many partitions can mean overfitting
    "model__max_leaf_nodes": [800, 1600, 3200, None],
}

In [ ]:
grid = GridSearchCV(
    estimator=train_pipe,
    param_grid=param_grid,
    scoring=(
        "f1_macro" if num_classes > 2
        else make_scorer(f1_score, pos_label="attack")
    ),
    n_jobs=4,
    refit=True,
    cv=4,
    verbose=3,
    return_train_score=True,
)

grid.fit(X_train, y_train)

trained_pipe = grid.best_estimator_
trained_model = trained_pipe["model"]

## Cross-validation results

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
cv_results.sort_values(by="rank_test_score", inplace=True)
cv_results.to_csv(results_dir / "train_cv_results.csv", index=False)
cv_results.head(10)

## Feature importance

In [ ]:
feature_names = trained_pipe["transformer"].get_feature_names_out()
feature_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": trained_pipe["model"].feature_importances_,
})
feature_importances.sort_values(
    by="importance", ascending=False,
    ignore_index=True, inplace=True,
)
feature_importances["importance_norm"] = minmax_scale(feature_importances["importance"])
feature_importances.to_csv(results_dir / "feature_importances.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 15))

feature_importances["importance"].plot(kind="barh", ax=ax)

ax.invert_yaxis()
ax.set_yticks(
    ticks=range(feature_importances.shape[0]),
    labels=feature_importances["feature"],
    fontsize=8,
)
ax.set_title("Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.savefig(results_dir / "feature_importances.png", dpi=300)
plt.show()

# Model evaluations

## Train-split evaluations

In [ ]:
y_proba = trained_pipe.predict_proba(X_train)

train_results = pd.DataFrame(
    data=y_proba,
    columns=trained_pipe.classes_,
    dtype="float32",
)

train_results.to_parquet(
    results_dir / "train_predict_proba.parquet",
    index=False,
    compression="gzip",
)

## Test-split evaluations

In [ ]:
y_proba = trained_pipe.predict_proba(X_test)

test_results = pd.DataFrame(
    data=y_proba,
    columns=trained_pipe.classes_,
    dtype="float32",
)

test_results.to_parquet(
    results_dir / "test_predict_proba.parquet",
    index=False,
    compression="gzip",
)

# Model summary and persistency

## Model persistency

In [ ]:
pipe_persist_path = results_dir / "pipe.pickle.gz"

with gz_open(pipe_persist_path, mode="wb") as pf:
    Pickler(pf, protocol=5).dump(trained_pipe)

## Model summary

In [ ]:
trained_tree = trained_model.tree_

model_summary = {
    "best_parameters": grid.best_params_,
    "all_parameters": trained_model.get_params(),
    "tree_structure": {
        "n_nodes": int(trained_tree.node_count),
        "n_leafs": int(trained_tree.n_leaves),
        "max_depth": int(trained_tree.max_depth),
    }
}

model_summary = dumps(model_summary, indent=2)
(results_dir / "model_summary.json").write_text(model_summary)
print("MODEL SUMMARY:", model_summary, sep="\n")